# Model Evaluation Notebook

This notebook evaluates CatBoost, XGBoost, and LightGBM models on the test set using F2-score, PR-AUC, Precision, Recall, and F1-score.

In [1]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import os
import gc
from pathlib import Path

# Set OpenMP threads for LightGBM/CatBoost on macOS
os.environ["OMP_NUM_THREADS"] = "1"

from sklearn.metrics import (
    precision_recall_curve, 
    average_precision_score, 
    f1_score, 
    fbeta_score, 
    precision_score, 
    recall_score, 
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from Models.AutoEncoder import AutoEncoder
from Utils.DataUtils import build_ae_dataloaders
from Utils.FeatureUtils import extract_features

# Set device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cpu


In [2]:
# Load AutoEncoder and Extract Features
checkpoint = torch.load("checkpoints/autoencoder.pt", map_location=DEVICE)

ae_model = AutoEncoder(
    input_dim=checkpoint["input_dim"],
    latent_dim=checkpoint["latent_dim"],
    hidden_dims=checkpoint["hidden_dims"],
).to(DEVICE)

ae_model.load_state_dict(checkpoint["model_state_dict"])
ae_model.eval()

_, _, test_loader, _ = build_ae_dataloaders(batch_size=1024, mode="ae")

print("Extracting features from test set...")
X_test, y_test = extract_features(ae_model, test_loader, DEVICE)
print(f"X_test shape: {X_test.shape}")

[INFO] Project root: /Users/leesean/DL_project
[INFO] Data dir: /Users/leesean/DL_project/data
[INFO] Loading train data from: /Users/leesean/DL_project/data/train_transaction.csv
[INFO] Loading test data from: /Users/leesean/DL_project/data/test_transaction.csv
[INFO] Train shape: (590540, 394)
[INFO] Test shape: (506691, 393)


/Users/leesean/DL_project/Utils/Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
/Users/leesean/DL_project/Utils/Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
/Users/leesean/DL_project/Utils/Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd

[INFO] Train: (472432, 776)
[INFO] Val: (59054, 776)
[INFO] Test: (59054, 776)
Extracting features from test set...
X_test shape: (59054, 1037)


In [3]:
def evaluate_model(y_true, y_probs, threshold=None, model_name="Model"):
    """
    Evaluates a model's performance on a test set.
    If threshold is None, it sweeps for the best F2 threshold.
    """
    pr_auc = average_precision_score(y_true, y_probs)
    roc_auc = roc_auc_score(y_true, y_probs)
    
    if threshold is None:
        precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)
        f2_scores = (5 * precisions * recalls) / (4 * precisions + recalls + 1e-10)
        best_f2_idx = np.argmax(f2_scores)
        threshold = thresholds[min(best_f2_idx, len(thresholds)-1)]
        print(f"Best F2 threshold for {model_name}: {threshold:.4f}")
    
    y_pred = (y_probs >= threshold).astype(int)
    
    metrics = {
        "Model": model_name,
        "PR-AUC": pr_auc,
        "ROC-AUC": roc_auc,
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1-Score": f1_score(y_true, y_pred),
        "F2-Score": fbeta_score(y_true, y_pred, beta=2),
        "Threshold": threshold
    }
    
    return metrics

In [4]:
# Load CatBoost
from catboost import CatBoostClassifier
import json

cb_model = CatBoostClassifier()
cb_model.load_model("checkpoints/best_catboost.cbm")

with open("checkpoints/catboost_data_checkpoint.json", "r") as f:
    cb_meta = json.load(f)
    cb_threshold = cb_meta.get("best_threshold", 0.5)

cb_probs = cb_model.predict_proba(X_test)[:, 1]
cb_metrics = evaluate_model(y_test, cb_probs, threshold=cb_threshold, model_name="CatBoost")
print("CatBoost Evaluation Done.")

CatBoost Evaluation Done.


In [5]:
display(cb_metrics)

{'Model': 'CatBoost',
 'PR-AUC': 0.7685650927229025,
 'ROC-AUC': 0.957531428807111,
 'Precision': 0.6665207877461706,
 'Recall': 0.7371732817037754,
 'F1-Score': 0.7000689496667433,
 'F2-Score': 0.721869371504408,
 'Threshold': 0.57}

In [6]:
# Load LightGBM
import lightgbm as lgb

lgb_booster = lgb.Booster(model_file="checkpoints/lgbm_model.txt")

with open("checkpoints/lightgbm_data_checkpoint.json", "r") as f:
    lgb_meta = json.load(f)
    lgb_threshold = lgb_meta.get("best_threshold", 0.5)

lgb_probs = lgb_booster.predict(X_test)
lgb_metrics = evaluate_model(y_test, lgb_probs, threshold=lgb_threshold, model_name="LightGBM")
print("LightGBM Evaluation Done.")

LightGBM Evaluation Done.


In [7]:
display(lgb_metrics)

{'Model': 'LightGBM',
 'PR-AUC': 0.7413917175291616,
 'ROC-AUC': 0.951848093764887,
 'Precision': 0.5907487259898079,
 'Recall': 0.7294288480154889,
 'F1-Score': 0.652804851635261,
 'F2-Score': 0.6967175219602404,
 'Threshold': 0.6149999999999999}

In [8]:
# Load XGBoost
import xgboost as xgb

xgb_model = xgb.XGBClassifier()
xgb_model.load_model("checkpoints/best_xgboost.json")

with open("checkpoints/xgboost_data_checkpoint.json", "r") as f:
    xgb_meta = json.load(f)
    xgb_threshold = xgb_meta.get("best_threshold", 0.5)

xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
xgb_metrics = evaluate_model(y_test, xgb_probs, threshold=xgb_threshold, model_name="XGBoost")
print("XGBoost Evaluation Done.")

XGBoost Evaluation Done.


In [9]:
display(xgb_metrics)

{'Model': 'XGBoost',
 'PR-AUC': 0.8019478056499406,
 'ROC-AUC': 0.9575864496464023,
 'Precision': 0.6554307116104869,
 'Recall': 0.7623426911907066,
 'F1-Score': 0.7048556724099351,
 'F2-Score': 0.7382581794318928,
 'Threshold': 0.09499999999999999}

In [10]:
from Models.FraudModel import FraudMLP

def load_gated_mlp(checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    config = checkpoint["model_config"]
    
    # Initialize encoder if needed
    encoder = None
    if config.get("has_encoder"):
        # Extract encoder architecture from state_dict keys/shapes
        # We already investigated these:
        # AE16: hidden [128, 64], latent 16
        # AE256: hidden [128, 64, 32], latent 256
        latent_dim = config["encoder_latent_dim"]
        if latent_dim == 16:
            hidden_dims = [128, 64]
        else:
            hidden_dims = [128, 64, 32]
            
        encoder = AutoEncoder(
            input_dim=config["input_dim"],
            latent_dim=latent_dim,
            hidden_dims=hidden_dims
        )
    
    model = FraudMLP(
        input_dim=config["input_dim"],
        hidden_dims=config["hidden_dims"],
        gated=config["gated"],
        dropout=config["dropout"],
        use_norm=config["use_norm"],
        encoder=encoder,
        freeze_encoder=config["freeze_encoder"],
        use_recon_error_vector=config.get("use_recon_error_vector", False)
    )
    
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()
    return model

gated_mlp_results = []
X_test_raw_tensor = torch.tensor(X_test[:, :776], dtype=torch.float32).to(DEVICE)

gated_mlp_models = [
    ("GatedMLP_AE16_V4", "checkpoints/GatedMLP_AE16_V4/best.pt", "checkpoints/GatedMLP_AE16_V4/summary.json"),
    ("GatedMLP_AE256_V1", "checkpoints/GatedMLP_AE256_V1/best.pt", "checkpoints/GatedMLP_AE256_V1/summary.json"),
    ("GatedMLP_V6", "checkpoints/GatedMLP_V6/best.pt", "checkpoints/GatedMLP_V6/summary.json"),
]

gated_mlp_probs = {}

for name, pt_path, json_path in gated_mlp_models:
    print(f"Evaluating {name}...")
    model = load_gated_mlp(pt_path, DEVICE)
    
    with open(json_path, "r") as f:
        summary = json.load(f)
        # Use best_threshold from val metrics if available
        threshold = summary.get("best_val_metrics", {}).get("best_threshold", 0.5)
    
    with torch.no_grad():
        logits = []
        # Batch inference for MLP to save memory
        for i in range(0, len(X_test_raw_tensor), 1024):
            batch = X_test_raw_tensor[i:i+1024]
            logits.append(model(batch))
        
        probs = torch.sigmoid(torch.cat(logits)).cpu().numpy()
        gated_mlp_probs[name] = probs
        
    metrics = evaluate_model(y_test, probs, threshold=threshold, model_name=name)
    gated_mlp_results.append(metrics)
    
    del model
    gc.collect()
    print(f"{name} Evaluation Done.")

all_mlp_metrics_df = pd.DataFrame(gated_mlp_results)
display(all_mlp_metrics_df)

Evaluating GatedMLP_AE16_V4...
GatedMLP_AE16_V4 Evaluation Done.
Evaluating GatedMLP_AE256_V1...
GatedMLP_AE256_V1 Evaluation Done.
Evaluating GatedMLP_V6...
GatedMLP_V6 Evaluation Done.


,Model,PR-AUC,ROC-AUC,Precision,Recall,F1-Score,F2-Score,Threshold
0,GatedMLP_AE16_V4,0.714250,0.920571,0.621397,0.688771,0.653352,0.674152,0.578400
1,GatedMLP_AE256_V1,0.687931,0.919753,0.602410,0.653437,0.626886,0.642551,0.695261
2,GatedMLP_V6,0.710398,0.926125,0.611667,0.680058,0.644052,0.665183,0.711950
